# RAG with PDF files and OCR

This demo provides an example where retrieval is performed on PDF files.

Note: To run the code with your own data, simply update the folder path (or the `folder_path` variable) accordingly.

In [1]:
try:
    !python -m pip install -e .

    # Install dependencies
    # !brew install -y tesseract
    %pip install -r requirements.txt
    
    print("setup complete")
except ImportError:
    print("Not running")

Obtaining file:///Users/pranavsehgal/Desktop/retrieval-engine-vector-rag/rag
  Attempting uninstall: RAG
    Found existing installation: RAG 1.0.0
    Uninstalling RAG-1.0.0:
      Successfully uninstalled RAG-1.0.0
  Running setup.py develop for RAG
You should consider upgrading via the '/Users/pranavsehgal/Desktop/retrieval-engine-vector-rag/.venv/bin/python -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/pranavsehgal/Desktop/retrieval-engine-vector-rag/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
setup complete


In [2]:
# Login to Hugging Face
import os
from dotenv import load_dotenv
from huggingface_hub import login, whoami

load_dotenv()
token = os.getenv("HF_TOKEN")

if token:
    print("Token loaded successfully!")
else:
    print("Token not found in the .env file")

login(token=token)

# Check if the login was successful 
try:
    user_info = whoami()
    print(f"Logged in successfully as: {user_info['name']}")
except Exception as e:
    print(f"Login failed: {str(e)}")

Token loaded successfully!


/Users/pranavsehgal/Desktop/retrieval-engine-vector-rag/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in successfully as: Pranav-Sehgal


In [3]:
%load_ext autoreload
%autoreload 2

### Import libraries

In [4]:
import os
import pytesseract
from src.PDFProcessor import PDFProcessor
from src.LargeLanguageModel import LargeLanguageModel
from src.IndexManager import IndexManager

# Configure pytesseract to use the Homebrew-installed tesseract
pytesseract.pytesseract.pytesseract_cmd = '/opt/homebrew/bin/tesseract'

### Set paths

In [5]:
folder_path = "pdf_files/"
index_path = 'resources/faiss_embeddings.index'

In [6]:
import os
import sys
# Add /opt/homebrew/bin to PATH so tesseract can be found
os.environ['PATH'] = '/opt/homebrew/bin:' + os.environ.get('PATH', '')

# Verify the configuration
import pytesseract
print(f"pytesseract_cmd: {pytesseract.pytesseract.pytesseract_cmd}")
print(f"PATH: {os.environ['PATH'][:100]}...")

# Test direct call
import subprocess
result = subprocess.run(['/opt/homebrew/bin/tesseract', '--version'], capture_output=True, text=True)
print(f"Direct tesseract call works: {result.returncode == 0}")

pytesseract_cmd: /opt/homebrew/bin/tesseract
PATH: /opt/homebrew/bin:/Users/pranavsehgal/Desktop/retrieval-engine-vector-rag/.venv/bin:/opt/homebrew/bi...
Direct tesseract call works: True


### Read PDF files from folder  
Extract text from PDF files in the specified folder (e.g., `folder_path = "pdf_files/"`)

In [7]:
texts, info = PDFProcessor.extract_text_from_pdfs_in_folder(folder_path)

Found 2 PDF files
Processing pdf_files/NIPS-2012-imagenet-classification-with-deep-convolutional-neural-networks-Paper.pdf
Processing pdf_files/1706.03762v7.pdf


### Split into smaller chunks for better retrieval

In [8]:
chunks, info = IndexManager.chunk_texts_and_info(texts, info)

### Create or Load the Embedding Index

Prepare dataset of embeddings from text for the retrieval. Build an index for efficient similarity search.

Find relevant text and then use an LLM to generate a response based on this text.

In [9]:
# Initialize LLM and IndexManager
llm = LargeLanguageModel(model_id="meta-llama/Llama-3.2-3B-Instruct")
index_manager = IndexManager(chunks, info, llm)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
if not os.path.isfile(index_path):
    print("Create embedding index")
    index = index_manager.create_index(texts,
                                       index_path)
else:
    print("Load embedding index")
    index = index_manager.load_index(index_path)

Load embedding index


### Query PDF documents

##### Retrieval

Retrieves the top-k relevant references, and constructs a prompt for the LLM

In [11]:
query = "What does this document say about attention mechanism?"
references = index_manager.query(text=query, top_k=5)

context = "\n".join([text for text, _ in references])
prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

##### Generate answer

In [12]:
answer = llm.decode(prompt)

print("Answer:")
print(answer)
print("\nReferences:")
for text_cur, info_cur in references:
    fname, page_num = info_cur.split()
    print(f"- {fname}, page {page_num}: {text_cur[:20]}")

Answer:
Context:
ﬁnal convolutional layer. This is because most of the net’s parameters are in the ﬁrst fully-connected layer, which takes the last convolutional layer as input. So to make the two nets have approximately the same number of parameters, we did not halve the size of the ﬁnal convolutional layer (nor the fully-conneced layers which follow). Therefore this comparison is biased in favor of the one-GPU net, since it is bigger than “half the size” of the two-GPU net. 3
Figure 2: An illustration of the architecture of our CNN, explicitly showing the delineation of responsibilities between the two GPUs. One GPU runs the layer-parts at the top of the ﬁgure while the other runs the layer-parts at the bottom. The GPUs communicate only at certain layers. The network’s input is 150,528-dimensional, and the number of neurons in the network’s remaining layers is given by 253,440–186,624–64,896–64,896–43,264– 4096–4096–1000. neurons in a kernel map). The second convolutional layer takes